load and check the dataset

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Paths are relative to the folder where the notebook is running.
raw_file = Path("data/raw/helpdesk_emails_200.csv")
output_folder = Path("data/processed")

if not raw_file.is_file():
    raise FileNotFoundError(
        f"Expected the CSV at {raw_file.resolve()}. "
        "Place it inside data/raw and run this cell again."
    )

emails = pd.read_csv(raw_file)

required_columns = ["subject", "body", "category", "scenario_id"]
missing_columns = set(required_columns) - set(emails.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

if emails[required_columns].isna().any().any():
    raise ValueError("Some rows have missing values. Check the original CSV.")

print("Total emails:", len(emails))
print("Unique scenarios:", emails["scenario_id"].nunique())
print("\nEmails per category:")
print(emails["category"].value_counts())

Total emails: 200
Unique scenarios: 40

Emails per category:
category
Network Issue       50
General Inquiry     50
Software Support    50
Account Access      50
Name: count, dtype: int64


Split the scenarios, then select their emails

In [2]:
# Each scenario must have one correct category across all its versions.
categories_per_scenario = emails.groupby("scenario_id")["category"].nunique()

if categories_per_scenario.gt(1).any():
    raise ValueError("At least one scenario has conflicting category labels.")

# Create a smaller table with one row per scenario.
scenarios = emails[["scenario_id", "category"]].drop_duplicates()

train_scenarios, test_scenarios = train_test_split(
    scenarios,
    test_size=0.20,
    stratify=scenarios["category"],
    random_state=42,
)

# Put every version of each scenario into its assigned dataset.
train_emails = emails[
    emails["scenario_id"].isin(train_scenarios["scenario_id"])
].copy()

test_emails = emails[
    emails["scenario_id"].isin(test_scenarios["scenario_id"])
].copy()

In [5]:
#Verify and save the split


train_ids = set(train_emails["scenario_id"])
test_ids = set(test_emails["scenario_id"])

# Check that no scenario appears on both sides and every row is included.
assert train_ids.isdisjoint(test_ids), "A scenario appears in both datasets."
assert len(train_emails) + len(test_emails) == len(emails), "Row count mismatch."

summary = pd.DataFrame({
    "Training emails": train_emails["category"].value_counts(),
    "Test emails": test_emails["category"].value_counts(),
}).sort_index()

display(summary)
print(f"Training: {len(train_emails)} emails from {len(train_ids)} scenarios")
print(f"Testing:  {len(test_emails)} emails from {len(test_ids)} scenarios")

output_folder.mkdir(parents=True, exist_ok=True)

train_emails.to_csv(output_folder / "train_emails.csv", index=False)
test_emails.to_csv(output_folder / "test_emails.csv", index=False)

print("Files saved in:", output_folder.resolve())

,Training emails,Test emails
category,,
Account Access,40,10
General Inquiry,40,10
Network Issue,40,10
Software Support,40,10


Training: 160 emails from 32 scenarios
Testing:  40 emails from 8 scenarios
Files saved in: C:\Users\msi\Desktop\email_triage_assistant\data\processed


In [6]:
#Define the classification rules


import re
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Read only the training data for this step.
train_emails = pd.read_csv(Path("data/processed/train_emails.csv"))

keywords = {
    "Account Access": [
        "password", "login", "log in", "sign in", "locked",
        "verification", "permission denied", "activation",
    ],
    "Network Issue": [
        "wifi", "internet", "network", "vpn", "connection",
        "signal", "wireless", "mobile data",
    ],
    "Software Support": [
        "install", "installer", "update", "crash", "compiler",
        "python", "spreadsheet", "plugin",
    ],
    "General Inquiry": [
        "opening hours", "what time", "how much", "where can",
        "borrow", "reserve", "policy", "booking",
    ],
}


def predict_baseline(subject, body):
    """Predict one category from the email subject and body."""
    text = f"{subject} {body}".lower()

    # Make "Wi-Fi", "Wi Fi", and "wifi" match the same keyword.
    text = re.sub(r"\bwi[\s-]?fi\b", "wifi", text)

    # Treat punctuation as spaces so "sign-in" also matches "sign in".
    text = " ".join(re.findall(r"[a-z0-9]+", text.splitlines()[0])) if False else " ".join(re.findall(r"[a-z0-9]+", text))

    scores = {}
    for category, terms in keywords.items():
        score = 0
        for term in terms:
            if f" {term} " in f" {text} ":
                score += 1
        scores[category] = score

    if max(scores.values()) == 0:
        return "General Inquiry"

    # For a tie, choose the first matching category in the dictionary.
    return max(scores, key=scores.get)

In [7]:
#Try one email before evaluating the classifier

example_subject = "Cannot sign in"
example_body = "I entered my password several times and now my account is locked."

prediction = predict_baseline(example_subject, example_body)
print("Predicted category:", prediction)

Predicted category: Account Access


In [8]:
#Measure the baseline and inspect its mistakes

baseline_results = train_emails.copy()

baseline_results["prediction"] = [
    predict_baseline(subject, body)
    for subject, body in zip(baseline_results["subject"], baseline_results["body"])
]

actual = baseline_results["category"]
predicted = baseline_results["prediction"]
categories = list(keywords)

print(f"Training accuracy: {accuracy_score(actual, predicted):.1%}")
print(
    "Training macro-F1:",
    round(f1_score(actual, predicted, labels=categories, average="macro"), 3),
)
print()
print(classification_report(actual, predicted, labels=categories, zero_division=0))

mistakes = baseline_results.loc[
    actual != predicted,
    ["scenario_id", "subject", "body", "category", "prediction"],
]

print("Misclassified training emails:", len(mistakes))
print("Distinct scenarios with mistakes:", mistakes["scenario_id"].nunique())

# Show one mistake per scenario so repeated versions do not fill the table.
display(mistakes.drop_duplicates("scenario_id").head(10))

Training accuracy: 84.4%
Training macro-F1: 0.845

                  precision    recall  f1-score   support

  Account Access       1.00      0.88      0.93        40
   Network Issue       0.89      1.00      0.94        40
Software Support       1.00      0.62      0.77        40
 General Inquiry       0.64      0.88      0.74        40

        accuracy                           0.84       160
       macro avg       0.88      0.84      0.85       160
    weighted avg       0.88      0.84      0.85       160

Misclassified training emails: 25
Distinct scenarios with mistakes: 5


,scenario_id,subject,body,category,prediction
1,S38,IT help please,Can visitors use campus Wi-Fi during our exhib...,General Inquiry,Network Issue
2,S21,Quick question,"Hi, Jupyter opens but the kernel dies as soon ...",Software Support,General Inquiry
8,S09,Help with shared folder,I can open the drive but I need edit access t...,Account Access,General Inquiry
9,S28,Quick question,"Hi, The slide editor says my presentation form...",Software Support,General Inquiry
10,S26,Meeting audio - follow up,"Hello IT team, The microphone works in the rec...",Software Support,General Inquiry


In [9]:
#Prepare the training data and model

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline

train_emails = pd.read_csv("data/processed/train_emails.csv")

# Model input: the words in the subject and body.
X = train_emails["subject"] + "\n" + train_emails["body"]

# Correct answer for each email.
y = train_emails["category"]

# Used only to keep versions of the same scenario together during evaluation.
groups = train_emails["scenario_id"]

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

In [10]:
#Compare both methods using the training scenarios

# In each round, whole scenarios are held back for validation.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X, y, groups))

# Verify that no scenario appears on both sides of any fold.
for fit_rows, validation_rows in folds:
    fit_groups = set(groups.iloc[fit_rows])
    validation_groups = set(groups.iloc[validation_rows])
    assert fit_groups.isdisjoint(validation_groups)

# Each email gets one prediction from a model trained on other scenarios.
ml_predictions = cross_val_predict(model, X, y, cv=folds)

# Apply the unchanged keyword rules from Cell 4 to the same emails.
baseline_predictions = [
    predict_baseline(subject, body)
    for subject, body in zip(train_emails["subject"], train_emails["body"])
]

comparison = pd.DataFrame([
    {
        "Method": name,
        "Correct out of 160": accuracy_score(y, predictions, normalize=False),
        "Accuracy (%)": round(100 * accuracy_score(y, predictions), 1),
        "Macro-F1": round(f1_score(y, predictions, average="macro"), 3),
    }
    for name, predictions in [
        ("Fixed keyword baseline", baseline_predictions),
        ("TF-IDF + Logistic Regression", ml_predictions),
    ]
])

display(comparison)

,Method,Correct out of 160,Accuracy (%),Macro-F1
0,Fixed keyword baseline,135.0,84.4,0.845
1,TF-IDF + Logistic Regression,92.0,57.5,0.538


In [11]:
#Fit the model on all training emails and try it


# After cross-validation, fit the model that you'll use for the final test.
model.fit(X, y)

subject = "Unable to access my account"
body = "I changed my password yesterday and now I cannot sign in."

prediction = model.predict([f"{subject}\n{body}"])[0]
print("Predicted category:", prediction)

Predicted category: Account Access


In [12]:
#Compare both methods on the 40 test emails

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

test_emails = pd.read_csv("data/processed/test_emails.csv")

# Verify that all test scenarios are new to both the model and its training data.
train_ids = set(train_emails["scenario_id"])
test_ids = set(test_emails["scenario_id"])
assert train_ids.isdisjoint(test_ids), "A test scenario also appears in training."

# Combine subject and body exactly as in Cell 7.
X_test = test_emails["subject"] + "\n" + test_emails["body"]
y_test = test_emails["category"]

# Use the model already fitted in Cell 9.
test_results = test_emails.copy()
test_results["model_prediction"] = model.predict(X_test)
test_results["baseline_prediction"] = [
    predict_baseline(subject, body)
    for subject, body in zip(test_emails["subject"], test_emails["body"])
]

labels = sorted(train_emails["category"].unique())
methods = {
    "Fixed keyword baseline": "baseline_prediction",
    "TF-IDF + Logistic Regression": "model_prediction",
}

rows = []
for method, column in methods.items():
    predictions = test_results[column]
    rows.append({
        "Method": method,
        "Correct": int((predictions == y_test).sum()),
        "Total": len(y_test),
        "Accuracy (%)": round(100 * accuracy_score(y_test, predictions), 1),
        "Macro-F1": round(
            f1_score(y_test, predictions, labels=labels,
                     average="macro", zero_division=0), 3
        ),
    })

test_comparison = pd.DataFrame(rows)
display(test_comparison)
print(f"Evaluated {len(test_emails)} emails from {len(test_ids)} scenarios.")

,Method,Correct,Total,Accuracy (%),Macro-F1
0,Fixed keyword baseline,35,40,87.5,0.867
1,TF-IDF + Logistic Regression,30,40,75.0,0.750


Evaluated 40 emails from 8 scenarios.


In [13]:
#See which categories each method confuses

for method, column in methods.items():
    matrix = confusion_matrix(y_test, test_results[column], labels=labels)

    print(method)
    display(pd.DataFrame(
        matrix,
        index=pd.Index(labels, name="Actual category"),
        columns=pd.Index(labels, name="Predicted category"),
    ))

Fixed keyword baseline


Predicted category,Account Access,General Inquiry,Network Issue,Software Support
Actual category,,,,
Account Access,10,0,0,0
General Inquiry,0,10,0,0
Network Issue,0,0,10,0
Software Support,0,5,0,5


TF-IDF + Logistic Regression


Predicted category,Account Access,General Inquiry,Network Issue,Software Support
Actual category,,,,
Account Access,10,0,0,0
General Inquiry,5,5,0,0
Network Issue,0,0,10,0
Software Support,5,0,0,5


In [14]:
#Select 10 genuine wrong predictions for error analysis

columns = ["scenario_id", "subject", "body", "category"]

# Held-out mistakes from Cell 10.
test_cases = test_results[columns].copy()
test_cases["prediction"] = test_results["model_prediction"]
test_cases["source"] = "Held-out test"

# Out-of-fold mistakes from Cell 8.
cv_cases = train_emails[columns].copy()
cv_cases["prediction"] = ml_predictions
cv_cases["source"] = "Training cross-validation"

all_cases = pd.concat([test_cases, cv_cases], ignore_index=True)
errors = all_cases.loc[
    all_cases["category"] != all_cases["prediction"]
].copy()

# Select different scenarios first.
review = errors.drop_duplicates("scenario_id").head(10).copy()

# If needed, fill the remaining places with other genuine wrong predictions.
if len(review) < 10:
    remaining = errors.drop(index=review.index)
    review = pd.concat([review, remaining.head(10 - len(review))])

review["likely_reason"] = ""
review["evidence_from_email"] = ""

results_folder = Path("results")
results_folder.mkdir(exist_ok=True)
review_file = results_folder / "model_error_review.csv"

# Preserve your written explanations if you rerun the notebook.
if not review_file.exists():
    review.to_csv(review_file, index=False)

with pd.option_context("display.max_colwidth", None):
    display(review[["source", "scenario_id", "body", "category", "prediction"]])

print("Error-review file:", review_file.resolve())

,source,scenario_id,body,category,prediction
1,Held-out test,S35,Is there an introduction to university IT services for new students?,General Inquiry,Account Access
12,Held-out test,S24,My document looks fine until I export it and the figures disappear.,Software Support,Account Access
40,Training cross-validation,S14,"Hi, The wireless signal disappears whenever I go upstairs in the hostel.",Network Issue,Software Support
41,Training cross-validation,S38,Can visitors use campus Wi-Fi during our exhibition? Could you help?,General Inquiry,Network Issue
45,Training cross-validation,S25,"Hi, Python says pandas is missing even though I installed it yesterday.",Software Support,Account Access
49,Training cross-validation,S28,"Hi, The slide editor says my presentation format is unsupported.",Software Support,Account Access
50,Training cross-validation,S26,"Hello IT team, The microphone works in the recorder but stays silent in the meeting app. Thanks.",Software Support,Network Issue
72,Training cross-validation,S29,My program builds on another PC but this IDE cannot find its compiler. Please advise.,Software Support,Account Access
73,Training cross-validation,S33,How much does a colour print cost at the library?,General Inquiry,Network Issue
74,Training cross-validation,S11,My laptop joins the campus Wi-Fi but no pages load in the library. Could you help?,Network Issue,General Inquiry


Error-review file: C:\Users\msi\Desktop\email_triage_assistant\results\model_error_review.csv
